<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day6_8_(260618)_Kafka_%EA%B8%B0%EB%B0%98_%EC%9E%90%EB%8F%99%EA%B2%B0%EC%A0%9C_%EC%84%B1%EA%B3%B5%C2%B7%EC%8B%A4%ED%8C%A8_%EC%9D%B4%EB%B2%A4%ED%8A%B8_%EC%B2%98%EB%A6%AC_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/Account.java

package com.example.autopayment.domain;

import jakarta.persistence.Entity;
import jakarta.persistence.GeneratedValue;
import jakarta.persistence.GenerationType;
import jakarta.persistence.Id;

@Entity
public class Account {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private Long userId;

    private Long balance;

    protected Account() {
    }

    public Account(Long userId, Long balance) {
        this.userId = userId;
        this.balance = balance;
    }

    public Long getId() {
        return id;
    }

    public Long getUserId() {
        return userId;
    }

    public Long getBalance() {
        return balance;
    }

    public boolean canPay(Long amount) {
        return balance >= amount;
    }

    public void withdraw(Long amount) {
        this.balance -= amount;
    }
}

Overwriting /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/Account.java


In [2]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/AutoPaymentRule.java

package com.example.autopayment.domain;

import jakarta.persistence.Entity;
import jakarta.persistence.GeneratedValue;
import jakarta.persistence.GenerationType;
import jakarta.persistence.Id;

@Entity
public class AutoPaymentRule {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private Long userId;

    private String merchantName;

    private Long amount;

    private String status;

    protected AutoPaymentRule() {
    }

    public AutoPaymentRule(Long userId, String merchantName, Long amount, String status) {
        this.userId = userId;
        this.merchantName = merchantName;
        this.amount = amount;
        this.status = status;
    }

    public Long getId() {
        return id;
    }

    public Long getUserId() {
        return userId;
    }

    public String getMerchantName() {
        return merchantName;
    }

    public Long getAmount() {
        return amount;
    }

    public String getStatus() {
        return status;
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/AutoPaymentRule.java


In [3]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/AutoPaymentHistory.java

package com.example.autopayment.domain;

import jakarta.persistence.Entity;
import jakarta.persistence.GeneratedValue;
import jakarta.persistence.GenerationType;
import jakarta.persistence.Id;

import java.time.LocalDateTime;

@Entity
public class AutoPaymentHistory {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private Long ruleId;

    private Long userId;

    private String merchantName;

    private Long amount;

    private String result;

    private String reason;

    private LocalDateTime createdAt;

    protected AutoPaymentHistory() {
    }

    public AutoPaymentHistory(Long ruleId, Long userId, String merchantName, Long amount, String result, String reason) {
        this.ruleId = ruleId;
        this.userId = userId;
        this.merchantName = merchantName;
        this.amount = amount;
        this.result = result;
        this.reason = reason;
        this.createdAt = LocalDateTime.now();
    }
}


Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/AutoPaymentHistory.java


In [4]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/PaymentEventLog.java

package com.example.autopayment.domain;

import jakarta.persistence.Column;
import jakarta.persistence.Entity;
import jakarta.persistence.Id;

import java.time.LocalDateTime;

@Entity
public class PaymentEventLog {

    @Id
    private String eventId;

    private String eventType;

    private String topicName;

    @Column(columnDefinition = "TEXT")
    private String payload;

    private LocalDateTime createdAt;

    protected PaymentEventLog() {
    }

    public PaymentEventLog(String eventId, String eventType, String topicName, String payload) {
        this.eventId = eventId;
        this.eventType = eventType;
        this.topicName = topicName;
        this.payload = payload;
        this.createdAt = LocalDateTime.now();
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/domain/PaymentEventLog.java


In [6]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AccountRepository.java

package com.example.autopayment.repository;

import com.example.autopayment.domain.Account;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.Optional;

public interface AccountRepository extends JpaRepository<Account, Long> {
    Optional<Account> findByUserId(Long userId);
}

Overwriting /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AccountRepository.java


In [7]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AutoPaymentRuleRepository.java

package com.example.autopayment.repository;

import com.example.autopayment.domain.AutoPaymentRule;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.List;

public interface AutoPaymentRuleRepository extends JpaRepository<AutoPaymentRule, Long> {
    List<AutoPaymentRule> findByStatus(String status);
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AutoPaymentRuleRepository.java


In [8]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AutoPaymentHistoryRepository.java

package com.example.autopayment.repository;

import com.example.autopayment.domain.AutoPaymentHistory;
import org.springframework.data.jpa.repository.JpaRepository;

public interface AutoPaymentHistoryRepository extends JpaRepository<AutoPaymentHistory, Long> {
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/AutoPaymentHistoryRepository.java


In [9]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/PaymentEventLogRepository.java

package com.example.autopayment.repository;

import com.example.autopayment.domain.PaymentEventLog;
import org.springframework.data.jpa.repository.JpaRepository;

public interface PaymentEventLogRepository extends JpaRepository<PaymentEventLog, String> {
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/repository/PaymentEventLogRepository.java


In [10]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/event/AutoPaymentRequestedEvent.java

package com.example.autopayment.event;

public record AutoPaymentRequestedEvent(
        String eventId,
        Long ruleId,
        Long userId,
        String merchantName,
        Long amount
) {
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/event/AutoPaymentRequestedEvent.java


In [11]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/event/AutoPaymentResultEvent.java

package com.example.autopayment.event;

public record AutoPaymentResultEvent(
        String eventId,
        Long ruleId,
        Long userId,
        String merchantName,
        Long amount,
        String result,
        String reason
) {
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/event/AutoPaymentResultEvent.java


In [12]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/config/KafkaTopicConfig.java

package com.example.autopayment.config;

import org.apache.kafka.clients.admin.NewTopic;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;

@Configuration
public class KafkaTopicConfig {

    public static final String REQUESTED = "auto-payment-requested";
    public static final String SUCCEEDED = "auto-payment-succeeded";
    public static final String FAILED = "auto-payment-failed";

    @Bean
    public NewTopic requestedTopic() {
        return new NewTopic(REQUESTED, 1, (short) 1);
    }

    @Bean
    public NewTopic succeededTopic() {
        return new NewTopic(SUCCEEDED, 1, (short) 1);
    }

    @Bean
    public NewTopic failedTopic() {
        return new NewTopic(FAILED, 1, (short) 1);
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/config/KafkaTopicConfig.java


In [13]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/kafka/AutoPaymentProducer.java

package com.example.autopayment.kafka;

import com.example.autopayment.config.KafkaTopicConfig;
import com.example.autopayment.event.AutoPaymentRequestedEvent;
import com.example.autopayment.event.AutoPaymentResultEvent;
import org.springframework.kafka.core.KafkaTemplate;
import org.springframework.stereotype.Component;

@Component
public class AutoPaymentProducer {

    private final KafkaTemplate<String, Object> kafkaTemplate;

    public AutoPaymentProducer(KafkaTemplate<String, Object> kafkaTemplate) {
        this.kafkaTemplate = kafkaTemplate;
    }

    public void sendRequested(AutoPaymentRequestedEvent event) {
        kafkaTemplate.send(KafkaTopicConfig.REQUESTED, String.valueOf(event.ruleId()), event);
    }

    public void sendSucceeded(AutoPaymentResultEvent event) {
        kafkaTemplate.send(KafkaTopicConfig.SUCCEEDED, String.valueOf(event.ruleId()), event);
    }

    public void sendFailed(AutoPaymentResultEvent event) {
        kafkaTemplate.send(KafkaTopicConfig.FAILED, String.valueOf(event.ruleId()), event);
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/kafka/AutoPaymentProducer.java


In [14]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/config/DataInitConfig.java

package com.example.autopayment.config;

import com.example.autopayment.domain.Account;
import com.example.autopayment.domain.AutoPaymentRule;
import com.example.autopayment.repository.AccountRepository;
import com.example.autopayment.repository.AutoPaymentRuleRepository;
import org.springframework.boot.CommandLineRunner;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;

@Configuration
public class DataInitConfig {

    @Bean
    CommandLineRunner initData(AccountRepository accountRepository,
                               AutoPaymentRuleRepository ruleRepository) {
        return args -> {
            accountRepository.save(new Account(1L, 100000L));
            ruleRepository.save(new AutoPaymentRule(1L, "통신비", 55000L, "ACTIVE"));
            ruleRepository.save(new AutoPaymentRule(1L, "보험료", 120000L, "ACTIVE"));
        };
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/config/DataInitConfig.java


In [15]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/service/AutoPaymentRunService.java

package com.example.autopayment.service;

import com.example.autopayment.event.AutoPaymentRequestedEvent;
import com.example.autopayment.kafka.AutoPaymentProducer;
import com.example.autopayment.repository.AutoPaymentRuleRepository;
import org.springframework.stereotype.Service;

import java.util.UUID;

@Service
public class AutoPaymentRunService {

    private final AutoPaymentRuleRepository ruleRepository;
    private final AutoPaymentProducer producer;

    public AutoPaymentRunService(AutoPaymentRuleRepository ruleRepository,
                                 AutoPaymentProducer producer) {
        this.ruleRepository = ruleRepository;
        this.producer = producer;
    }

    public String run() {
        var rules = ruleRepository.findByStatus("ACTIVE");

        for (var rule : rules) {
            var event = new AutoPaymentRequestedEvent(
                    UUID.randomUUID().toString(),
                    rule.getId(),
                    rule.getUserId(),
                    rule.getMerchantName(),
                    rule.getAmount()
            );

            producer.sendRequested(event);
        }

        return "AUTO_PAYMENT_REQUESTED_COUNT=" + rules.size();
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/service/AutoPaymentRunService.java


In [16]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/kafka/AutoPaymentConsumer.java

package com.example.autopayment.kafka;

import com.example.autopayment.config.KafkaTopicConfig;
import com.example.autopayment.domain.AutoPaymentHistory;
import com.example.autopayment.domain.PaymentEventLog;
import com.example.autopayment.event.AutoPaymentRequestedEvent;
import com.example.autopayment.event.AutoPaymentResultEvent;
import com.example.autopayment.repository.AccountRepository;
import com.example.autopayment.repository.AutoPaymentHistoryRepository;
import com.example.autopayment.repository.PaymentEventLogRepository;
import com.fasterxml.jackson.core.JsonProcessingException;
import com.fasterxml.jackson.databind.ObjectMapper;
import org.springframework.kafka.annotation.KafkaListener;
import org.springframework.stereotype.Component;
import org.springframework.transaction.annotation.Transactional;

import java.util.UUID;

@Component
public class AutoPaymentConsumer {

    private final AccountRepository accountRepository;
    private final AutoPaymentHistoryRepository historyRepository;
    private final PaymentEventLogRepository eventLogRepository;
    private final AutoPaymentProducer producer;
    private final ObjectMapper objectMapper;

    public AutoPaymentConsumer(AccountRepository accountRepository,
                               AutoPaymentHistoryRepository historyRepository,
                               PaymentEventLogRepository eventLogRepository,
                               AutoPaymentProducer producer,
                               ObjectMapper objectMapper) {
        this.accountRepository = accountRepository;
        this.historyRepository = historyRepository;
        this.eventLogRepository = eventLogRepository;
        this.producer = producer;
        this.objectMapper = objectMapper;
    }

    @Transactional
    @KafkaListener(topics = KafkaTopicConfig.REQUESTED, groupId = "auto-payment-processor")
    public void consumeRequested(AutoPaymentRequestedEvent event) {
        saveEventLog(event.eventId(), "AutoPaymentRequested", KafkaTopicConfig.REQUESTED, event);

        var account = accountRepository.findByUserId(event.userId())
                .orElseThrow(() -> new IllegalStateException("ACCOUNT_NOT_FOUND"));

        if (account.canPay(event.amount())) {
            account.withdraw(event.amount());

            historyRepository.save(new AutoPaymentHistory(
                    event.ruleId(),
                    event.userId(),
                    event.merchantName(),
                    event.amount(),
                    "SUCCESS",
                    null
            ));

            var resultEvent = new AutoPaymentResultEvent(
                    UUID.randomUUID().toString(),
                    event.ruleId(),
                    event.userId(),
                    event.merchantName(),
                    event.amount(),
                    "SUCCESS",
                    null
            );

            producer.sendSucceeded(resultEvent);
            saveEventLog(resultEvent.eventId(), "AutoPaymentSucceeded", KafkaTopicConfig.SUCCEEDED, resultEvent);
        } else {
            historyRepository.save(new AutoPaymentHistory(
                    event.ruleId(),
                    event.userId(),
                    event.merchantName(),
                    event.amount(),
                    "FAILED",
                    "INSUFFICIENT_BALANCE"
            ));

            var resultEvent = new AutoPaymentResultEvent(
                    UUID.randomUUID().toString(),
                    event.ruleId(),
                    event.userId(),
                    event.merchantName(),
                    event.amount(),
                    "FAILED",
                    "INSUFFICIENT_BALANCE"
            );

            producer.sendFailed(resultEvent);
            saveEventLog(resultEvent.eventId(), "AutoPaymentFailed", KafkaTopicConfig.FAILED, resultEvent);
        }
    }

    private void saveEventLog(String eventId, String eventType, String topicName, Object event) {
        try {
            String payload = objectMapper.writeValueAsString(event);
            eventLogRepository.save(new PaymentEventLog(eventId, eventType, topicName, payload));
        } catch (JsonProcessingException e) {
            throw new IllegalStateException(e);
        }
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/kafka/AutoPaymentConsumer.java


In [17]:
%%writefile /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/controller/AutoPaymentController.java

package com.example.autopayment.controller;

import com.example.autopayment.repository.AccountRepository;
import com.example.autopayment.repository.AutoPaymentHistoryRepository;
import com.example.autopayment.repository.PaymentEventLogRepository;
import com.example.autopayment.service.AutoPaymentRunService;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.RestController;

import java.util.Map;

@RestController
public class AutoPaymentController {

    private final AutoPaymentRunService runService;
    private final AccountRepository accountRepository;
    private final AutoPaymentHistoryRepository historyRepository;
    private final PaymentEventLogRepository eventLogRepository;

    public AutoPaymentController(AutoPaymentRunService runService,
                                 AccountRepository accountRepository,
                                 AutoPaymentHistoryRepository historyRepository,
                                 PaymentEventLogRepository eventLogRepository) {
        this.runService = runService;
        this.accountRepository = accountRepository;
        this.historyRepository = historyRepository;
        this.eventLogRepository = eventLogRepository;
    }

    @PostMapping("/api/auto-payments/run")
    public String runAutoPayments() {
        return runService.run();
    }

    @GetMapping("/api/auto-payments/status")
    public Map<String, Object> status() {
        return Map.of(
                "accounts", accountRepository.findAll(),
                "histories", historyRepository.findAll(),
                "eventLogs", eventLogRepository.findAll()
        );
    }
}

Writing /content/kafka-auto-payment-spring-lab/auto-payment-api/src/main/java/com/example/autopayment/controller/AutoPaymentController.java
